# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
import json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms
from IPython.core.display import Markdown
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    citations: list = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        if (d := norm_func(ev)) is not None: yield d

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### PartAccum

Collator for tool calls and other text parts

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt='', citations=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt, data=dict(citations=citations or []))
        else:
            if typ==PartType.tool_use:
                    new_args = tc_kwargs.get('arguments', '')
                    cur_args = self.parts[index].arguments
                    if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                    elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                    else: self.parts[index].arguments = new_args
            else: 
                self.parts[index].text += txt
                # anthropic citations have matching idx
                self.parts[index].data['citations'].extend(citations or [])
    
    def finalize(self):
        for idx,tc in self.parts.items():
            if isinstance(tc, ToolCall):
                if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments) if tc.arguments else {}
                self.tool_calls.append(tc)
                self.parts[idx] = Part(type=PartType.tool_use, data={**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server})
        
        merged = []
        for p in self.parts.values():
            if merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking): merged[-1].text += p.text
            else: merged.append(p)        
        self.parts = merged

### _accolect_stream

A generic `_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
#| export
async def _acollect_stream(it, index_fn, model=None, api_name=None, vendor_name=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum = PartAccum()
    deltas = []
    typ, last_typ, last_idx, last_idx = None, None, None, None
    async for d in it:
        if d.text:     
            yield {'text': d.text}
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.text)
        if d.thinking: 
            yield {'thinking': d.thinking}
            typ = PartType.thinking
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.thinking)
        if d.citations:
            yield {'citations': d.citations} # anthropic
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, citations=d.citations)
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', '') if '_delta' in tc.arguments else tc.arguments
            tc_kwargs = dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra)
            typ = PartType.tool_use
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, **tc_kwargs)
        if d.server_tool_result: 
            typ = PartType.server_tool_result
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if d.refusal:
            typ = PartType.refusal
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.refusal)        
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
    part_accum.finalize()
    # need to recheck for tool calls post collation for streaming
    tcs = part_accum.tool_calls
    fin = FinishReason.tool_calls if fin==FinishReason.stop and any(~L(tcs).attrgot('server')) else fin
    # tool calls and non-anthropic citations are yielded at the end
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin,
            usage=usg,
            tool_calls=tcs,
            api_name=api_name,
            vendor_name=vendor_name,
            raw={'deltas':deltas})

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()